# Proyecto Final Mecánica Celeste
Soleil Dayana Niño Murcia 1033097666

Este proyecto del curso de Mecánica Celeste reta a los estudiantes a investigar la histórica aproximación del asteroide (99942) Apophis a la Tierra en abril de 2029. Asumiendo el rol de investigadores, los alumnos deberán aplicar la teoría, los métodos y los algoritmos vistos en clase para analizar exhaustivamente la dinámica de este evento astronómico, explorando opciones como la integración de N cuerpos, el cálculo de distancias mínimas o las alteraciones en su órbita heliocéntrica. El trabajo se debe desarrollar de forma estrictamente individual, culminando en la entrega de un Jupyter Notebook completamente ejecutado y argumentado, el cual deberá estar alojado en un repositorio de GitHub que documente todos los códigos, experimentos numéricos y conclusiones científicas obtenidas

In [15]:
%pip install -Uq pymcel

Note: you may need to restart the kernel to use updated packages.


In [16]:
# Librerías
import pymcel as pc
import numpy as np
import plotly.graph_objects as go

## Sobre Rebound y cómo funciona
**agregar contexto e importancia e implementación de Rebound**

In [17]:
%pip install -Uq rebound

Note: you may need to restart the kernel to use updated packages.


In [18]:
import rebound as rb

### Ejemplo de uso de Rebound

In [19]:
sim_solar = rb.Simulation()
sim_solar.add('Sun', date = '2024-01-01')
sim_solar.add('Earth', date = '2024-01-01')
sim_solar.G = 1  # Unidades canónicas

Searching NASA Horizons for 'Sun'... 
Found: Sun (10) 
Searching NASA Horizons for 'Earth'... 
Found: Earth-Moon Barycenter (3) (chosen from query 'Earth')


### Creación del primer sistema: Sol, Luna, Tierra, Jupiter, Apophis.

Para iniciar nuestras pruebas, crearemos un sistema que incluya el Sol, la Tierra, la Luna y Júpiter ** justificar la decisión*

In [20]:
# Sistema Sol-Tierra-Luna-Júpiter-Apophis con condiciones iniciales en 2028-01-01
sim_solar = rb.Simulation()

# IDs de NASA Horizons para evitar ambigüedad en nombres
cuerpos = [
    ("Sun", "10"),
    ("Earth", "399"),
    ("Moon", "301"),
    ("Jupiter", "599"),
    ("Apophis", "99942"),
]

for _, consulta in cuerpos:
    sim_solar.add(consulta, date="2028-01-01")

sim_solar.integrator = "ias15"
sim_solar.move_to_com()

Searching NASA Horizons for '10'... 
Found: Sun (10) 
Searching NASA Horizons for '399'... 
Found: Earth (399) 
Searching NASA Horizons for '301'... 
Found: Moon (301) 
Searching NASA Horizons for '599'... 
Found: Jupiter (599) 
Searching NASA Horizons for '99942'... 
Found: 99942 Apophis (2004 MN4) 


/usr/local/python/3.12.1/lib/python3.12/site-packages/rebound/horizons.py:184: RuntimeWarning: Warning: Mass cannot be retrieved from NASA HORIZONS. Set to 0.
  warnings.warn("Warning: Mass cannot be retrieved from NASA HORIZONS. Set to 0.", RuntimeWarning)


In [21]:
print("Sistema creado correctamente con:", [n for n, _ in cuerpos])

Sistema creado correctamente con: ['Sun', 'Earth', 'Moon', 'Jupiter', 'Apophis']


Integración:

In [22]:
sim_solar.integrate(sim_solar.t + 2*np.pi)  # Integrar un año canónico (2*pi)

In [23]:
# Integración análoga para los 5 cuerpos (marco inercial y marco del centro de masa)
nb = len(cuerpos) # número de cuerpos
ts = np.linspace(sim_solar.t, sim_solar.t + 10.0, 100)  # avanzar en el tiempo desde el estado actual
nt = len(ts) # número de pasos de tiempo

rs = np.zeros((nb, nt, 3))
vs = np.zeros((nb, nt, 3))
rps = np.zeros((nb, nt, 3))
vps = np.zeros((nb, nt, 3))

for i, t in enumerate(ts):
    sim_solar.integrate(float(t))

    # Marco inercial
    for j in range(nb):
        rs[j, i] = sim_solar.particles[j].xyz
        vs[j, i] = sim_solar.particles[j].vxyz

    # Marco del centro de masa
    sim_solar.move_to_com()
    for j in range(nb):
        rps[j, i] = sim_solar.particles[j].xyz
        vps[j, i] = sim_solar.particles[j].vxyz

print("Integración completada para:", [nombre for nombre, _ in cuerpos])

Integración completada para: ['Sun', 'Earth', 'Moon', 'Jupiter', 'Apophis']


Sabemos previamente de la teoría que el asteroide impactará en 2029, para iniciar esta prueba, integremos en un rango corto de 2028-2029 (pretendiendo próximamente calcular desde la fecha actual)

In [24]:
# Integración robusta desde 2028-01-01 hasta mediados de 2029 (1.5 años ≈ 548 días)
sim_solar = rb.Simulation()
for _, consulta in cuerpos:
    sim_solar.add(consulta, date="2028-01-01")
sim_solar.integrator = "ias15"
sim_solar.move_to_com()

nombres = [nombre for nombre, _ in cuerpos]
nb = len(nombres)

t0 = float(sim_solar.t)
duracion_dias = int(round(1.5 * 365.25))
ts = np.linspace(t0, t0 + duracion_dias, duracion_dias + 1)
nt = len(ts)

rs = np.zeros((nb, nt, 3))
vs = np.zeros((nb, nt, 3))
rps = np.zeros((nb, nt, 3))
vps = np.zeros((nb, nt, 3))

for i, t_abs in enumerate(ts):
    sim_solar.integrate(float(t_abs))

    # Marco inercial
    for k in range(nb):
        p = sim_solar.particles[k]
        rs[k, i] = p.xyz
        vs[k, i] = p.vxyz

    # Marco baricéntrico (sin alterar el estado de integración)
    masas = np.array([sim_solar.particles[k].m for k in range(nb)])
    r_cm = np.einsum("k,kj->j", masas, rs[:, i, :]) / masas.sum()
    v_cm = np.einsum("k,kj->j", masas, vs[:, i, :]) / masas.sum()
    rps[:, i, :] = rs[:, i, :] - r_cm
    vps[:, i, :] = vs[:, i, :] - v_cm

trayectorias = {n: rps[k].copy() for k, n in enumerate(nombres)}

print(f"Integración completada: {duracion_dias} días desde 2028-01-01.")

Searching NASA Horizons for '10'... 


Found: Sun (10) 
Searching NASA Horizons for '399'... 
Found: Earth (399) 
Searching NASA Horizons for '301'... 
Found: Moon (301) 
Searching NASA Horizons for '599'... 
Found: Jupiter (599) 
Searching NASA Horizons for '99942'... 
Found: 99942 Apophis (2004 MN4) 
Integración completada: 548 días desde 2028-01-01.


In [25]:
# Verificación de NaN y distancia mínima Tierra-Apophis
if "Earth" not in trayectorias or "Apophis" not in trayectorias:
    raise ValueError("No se encontraron 'Earth' y 'Apophis' en trayectorias.")

print("Chequeo de finitud por cuerpo:")
for n in nombres:
    xyz = trayectorias[n]
    finite_mask = np.isfinite(xyz).all(axis=1)
    n_nan = int((~finite_mask).sum())
    print(f"- {n:8s}: pasos válidos={finite_mask.sum():3d}/{len(finite_mask)} | NaN={n_nan}")

r_earth = trayectorias["Earth"]
r_apophis = trayectorias["Apophis"]

distancia_tierra_apophis = np.linalg.norm(r_earth - r_apophis, axis=1)
mascara_dist = np.isfinite(distancia_tierra_apophis)
if not mascara_dist.any():
    raise ValueError("Todas las distancias Tierra-Apophis son NaN.")

indices_validos = np.where(mascara_dist)[0]
idx_min = indices_validos[np.argmin(distancia_tierra_apophis[mascara_dist])]
dmin_au = float(distancia_tierra_apophis[idx_min])
dmin_km = dmin_au * 149597870.7

t_min_dias = float(ts[idx_min] - ts[0])
t_min_anios = t_min_dias / 365.25
fecha_ref = np.datetime64("2028-01-01")
fecha_min_aprox = fecha_ref + np.timedelta64(int(round(t_min_dias)), "D")

print("\nDistancia mínima Tierra-Apophis:")
print(f"- índice: {idx_min}")
print(f"- t (días desde 2028-01-01): {t_min_dias:.2f}")
print(f"- t (años desde 2028-01-01): {t_min_anios:.6f}")
print(f"- fecha aproximada: {fecha_min_aprox}")
print(f"- distancia mínima: {dmin_au:.6e} UA ({dmin_km:,.1f} km)")

if idx_min in (0, len(ts)-1):
    print("Aviso: el mínimo cayó en un borde del intervalo; conviene ampliar la ventana temporal.")

Chequeo de finitud por cuerpo:
- Sun     : pasos válidos=549/549 | NaN=0
- Earth   : pasos válidos=549/549 | NaN=0
- Moon    : pasos válidos=549/549 | NaN=0
- Jupiter : pasos válidos=549/549 | NaN=0
- Apophis : pasos válidos=549/549 | NaN=0

Distancia mínima Tierra-Apophis:
- índice: 8
- t (días desde 2028-01-01): 8.00
- t (años desde 2028-01-01): 0.021903
- fecha aproximada: 2028-01-09
- distancia mínima: 1.332329e-02 UA (1,993,135.1 km)


In [26]:
# Gráfica de distancia Tierra-Apophis y marca del mínimo
ts_rel_dias = ts - ts[0]
fechas = [np.datetime64("2028-01-01") + np.timedelta64(int(round(float(d))), "D") for d in ts_rel_dias]

fig_d = go.Figure()
fig_d.add_trace(
    go.Scatter(
        x=fechas,
        y=distancia_tierra_apophis,
        mode="lines",
        name="Distancia Tierra-Apophis [UA]",
        line=dict(color="royalblue", width=2),
    )
)

fig_d.add_trace(
    go.Scatter(
        x=[fechas[idx_min]],
        y=[distancia_tierra_apophis[idx_min]],
        mode="markers",
        name="Mínimo",
        marker=dict(color="crimson", size=10, symbol="diamond"),
    )
)

fig_d.update_layout(
    title="Distancia Tierra-Apophis en el tiempo",
    xaxis_title="Fecha",
    yaxis_title="Distancia [UA]",
    template="plotly_white",
)

fig_d.show()

In [27]:
fig = go.Figure()

for n in nombres:
    xyz = trayectorias[n]
    mask = np.isfinite(xyz).all(axis=1)
    xyz_ok = xyz[mask]

    if xyz_ok.size == 0:
        continue

    es_apophis = (n == "Apophis")
    color = "crimson" if es_apophis else "rgba(120,120,120,0.65)"
    width = 6 if es_apophis else 2

    fig.add_trace(
        go.Scatter3d(
            x=xyz_ok[:, 0],
            y=xyz_ok[:, 1],
            z=xyz_ok[:, 2],
            mode="lines",
            name=n,
            line=dict(color=color, width=width),
            opacity=1.0 if es_apophis else 0.55,
        )
    )

    if es_apophis:
        fig.add_trace(
            go.Scatter3d(
                x=[xyz_ok[0, 0], xyz_ok[-1, 0]],
                y=[xyz_ok[0, 1], xyz_ok[-1, 1]],
                z=[xyz_ok[0, 2], xyz_ok[-1, 2]],
                mode="markers",
                name="Apophis (inicio/fin)",
                marker=dict(size=[6, 9], color=["orange", "red"], symbol=["circle", "diamond"]),
            )
        )

fig.update_layout(
    title="Trayectorias 3D del sistema (énfasis en Apophis)",
    scene=dict(
        xaxis_title="x [UA]",
        yaxis_title="y [UA]",
        zaxis_title="z [UA]",
        aspectmode="data",
    ),
    legend=dict(itemsizing="constant"),
    template="plotly_white",
)

if len(fig.data) == 0:
    print("No hay puntos válidos para graficar (las trayectorias parecen contener NaN).")
else:
    fig.show()